# SPY Options Chain Analysis — 04/06/2026 Expiry
Generated by autonomous data-analyst agent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('spy_options_clean.csv')
df_calls = df[df['Type'] == 'Call'].sort_values('Strike').reset_index(drop=True)
df_puts  = df[df['Type'] == 'Put'].sort_values('Strike').reset_index(drop=True)
S = 628.0   # approximate SPY spot price on 4/3/2026 close
r = 0.045   # risk-free rate
T = 3/365   # days to expiry: 04/06/2026 (3 trading days)
print(f"Loaded {len(df)} options ({len(df_calls)} calls, {len(df_puts)} puts)")
print(df.head())

In [ ]:
def d1(S, K, T, r, sigma):
    return (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def calc_delta(S, K, T, r, sigma, opt_type):
    val = d1(S, K, T, r, sigma)
    if opt_type == 'Call':
        return norm.cdf(val)
    return norm.cdf(val) - 1

def calc_gamma(S, K, T, r, sigma):
    return norm.pdf(d1(S, K, T, r, sigma)) / (S * sigma * np.sqrt(T))

df_calls['Delta'] = df_calls.apply(lambda row: calc_delta(S, row['Strike'], T, r, row['IV'], 'Call'), axis=1)
df_calls['Gamma'] = df_calls.apply(lambda row: calc_gamma(S, row['Strike'], T, r, row['IV']), axis=1)
df_puts['Delta']  = df_puts.apply(lambda row: calc_delta(S, row['Strike'], T, r, row['IV'], 'Put'), axis=1)
df_puts['Gamma']  = df_puts.apply(lambda row: calc_gamma(S, row['Strike'], T, r, row['IV']), axis=1)

atm_call = df_calls.iloc[(df_calls['Strike'] - S).abs().argsort().iloc[0]]
atm_iv = atm_call['IV']
move_1sd = S * atm_iv * np.sqrt(T)
sd_levels = {
    '1 SD':   (S - move_1sd,       S + move_1sd),
    '2 SD':   (S - 2 * move_1sd,   S + 2 * move_1sd),
    '2.5 SD': (S - 2.5 * move_1sd, S + 2.5 * move_1sd)
}
print(f'ATM IV: {atm_iv:.2%}, 1SD move: +/-{move_1sd:.2f} pts')
for k, (lo, hi) in sd_levels.items():
    print(f'{k}: [{lo:.2f}, {hi:.2f}]')

In [ ]:
def plot_sd_bands(ax):
    colors = {'1 SD': 'green', '2 SD': 'orange', '2.5 SD': 'red'}
    for label, (lower, upper) in sd_levels.items():
        ax.axvline(lower, color=colors[label], linestyle='--', alpha=0.6, linewidth=1)
        ax.axvline(upper, color=colors[label], linestyle='--', alpha=0.6, linewidth=1, label=f'{label}: [{lower:.1f}, {upper:.1f}]')
    ax.axvline(S, color='black', linestyle='-', linewidth=1.5, label=f'Spot ({S})')

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 15), sharex=True)

ax1.plot(df_calls['Strike'], df_calls['IV'] * 100, marker='o', color='steelblue', linewidth=2, label='Call IV')
ax1.plot(df_puts['Strike'],  df_puts['IV']  * 100, marker='s', color='salmon',    linewidth=2, label='Put IV')
ax1.set_title('SPY Implied Volatility Smile — 04/06/2026 Expiry', fontsize=13, fontweight='bold')
ax1.set_ylabel('Implied Volatility (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)
plot_sd_bands(ax1)

ax2.plot(df_calls['Strike'], df_calls['Delta'], marker='o', color='purple', linewidth=2, label='Call Delta')
ax2.plot(df_puts['Strike'],  df_puts['Delta'],  marker='s', color='teal',   linewidth=2, label='Put Delta')
ax2.set_title('Delta by Strike', fontsize=13, fontweight='bold')
ax2.set_ylabel('Delta')
ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='Call ATM (0.5)')
ax2.axhline(-0.5, color='gray', linestyle=':', alpha=0.7, label='Put ATM (-0.5)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plot_sd_bands(ax2)

ax3.plot(df_calls['Strike'], df_calls['Gamma'], marker='o', color='crimson',  linewidth=2, label='Call Gamma')
ax3.plot(df_puts['Strike'],  df_puts['Gamma'],  marker='s', color='darkorange', linewidth=2, label='Put Gamma')
ax3.set_title('Gamma Risk Profile', fontsize=13, fontweight='bold')
ax3.set_xlabel('Strike Price')
ax3.set_ylabel('Gamma')
ax3.legend()
ax3.grid(True, alpha=0.3)
plot_sd_bands(ax3)

fig.suptitle('SPY Options Analysis — 04/06/2026', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('spy_options_analysis.png', dpi=150, bbox_inches='tight')
print('Saved spy_options_analysis.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(df_calls['Strike'], df_calls['OI'], color='steelblue', alpha=0.8, width=1)
axes[0].set_title('Call Open Interest by Strike', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Strike')
axes[0].set_ylabel('Open Interest')
axes[0].grid(True, alpha=0.3)
for label, (lo, hi) in sd_levels.items():
    axes[0].axvline(lo, color='red', linestyle='--', alpha=0.5, linewidth=1)
    axes[0].axvline(hi, color='red', linestyle='--', alpha=0.5, linewidth=1)
axes[0].axvline(S, color='black', linestyle='-', linewidth=1.5, label=f'Spot ({S})')
axes[0].legend()

axes[1].bar(df_puts['Strike'], df_puts['OI'], color='salmon', alpha=0.8, width=1)
axes[1].set_title('Put Open Interest by Strike', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Strike')
axes[1].set_ylabel('Open Interest')
axes[1].grid(True, alpha=0.3)
for label, (lo, hi) in sd_levels.items():
    axes[1].axvline(lo, color='red', linestyle='--', alpha=0.5, linewidth=1)
    axes[1].axvline(hi, color='red', linestyle='--', alpha=0.5, linewidth=1)
axes[1].axvline(S, color='black', linestyle='-', linewidth=1.5, label=f'Spot ({S})')
axes[1].legend()

plt.tight_layout()
plt.savefig('oi_by_strike.png', dpi=150, bbox_inches='tight')
print('Saved oi_by_strike.png')

In [ ]:
print('=== CALL SUMMARY ===')
print(df_calls[['Strike', 'Last', 'Bid', 'Ask', 'IV', 'Delta', 'Gamma', 'OI']].to_string(index=False))
print()
print('=== PUT SUMMARY ===')
print(df_puts[['Strike', 'Last', 'Bid', 'Ask', 'IV', 'Delta', 'Gamma', 'OI']].to_string(index=False))
print()
print(f'ATM IV: {atm_iv:.4f} ({atm_iv:.2%})')
print(f'Expected 1SD move (3d): +/- {move_1sd:.2f} pts')
for k, (lo, hi) in sd_levels.items():
    print(f'{k}: [{lo:.2f}, {hi:.2f}]')
print(f'Max call OI strike: {df_calls.loc[df_calls["OI"].idxmax(), "Strike"]}')
print(f'Max put  OI strike: {df_puts.loc[df_puts["OI"].idxmax(), "Strike"]}')